### notebook 03
- designing evaluation splits and leakage tests

goals:
- check target identifiers and repeated observations from ribes-labeled dataset before split
- create validation/test assignments for:
  - molecule-grouped random split (group same structures together)
  - held-out target split (keep each uniprot target id entirely within one partition)
- check partition sizes, class balance, and target/molecule overlap
- save split assignments to csv in /processed

In [2]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

project_root = Path.cwd().parent

### 0. load ribes-labeled dataset

In [3]:
data_path = project_root / "data" / "processed" / "protac_clean.csv"

protac_clean = pd.read_csv(data_path)
display(protac_clean.head())

# verify the starting dataset
print("shape:", protac_clean.shape)
print("\nlabel counts:")
print(protac_clean["Label"].value_counts())
print("missing uniprot id:", protac_clean["Uniprot"].isna().sum())

,Compound ID,Article DOI,Canonicalised Smiles,Target,Uniprot,E3 ligase,DC50 (nM),Dmax (%),Assay (DC50/Dmax),Ribes Label,Label
0,11,10.1021/acs.jmedchem.8b01413,COc1cc(-c2cn(C)c(=O)c3cnccc23)cc(OC)c1CN1CCN(C...,BRD9,Q9H8M2,VHL,560,80,Degradation of BRD9 in HeLa cells after 4 h tr...,positive,1
1,22,10.1021/acs.jmedchem.8b01413,COc1cc(-c2cn(C)c(=O)c3cnccc23)cc(OC)c1CN1CCN(C...,BRD9,Q9H8M2,VHL,1.76,95,Degradation of BRD9 in RI-1 cells after 8 h tr...,positive,1
2,22,10.1021/acs.jmedchem.8b01413,COc1cc(-c2cn(C)c(=O)c3cnccc23)cc(OC)c1CN1CCN(C...,BRD7,Q9NPI1,VHL,4.5,95,Degradation of BRD7 in RI-1 cells after 8 h tr...,positive,1
3,151,10.1038/s41467-023-39904-5,Cc1ncsc1-c1ccc(CNC(=O)[C@@H]2C[C@@H](O)CN2C(=O...,BRD4,O60885,VHL,518,91,Degradation of BRD4 in A375 cells after 4 h tr...,positive,1
4,152,10.1038/s41467-023-39904-5,Cc1ncsc1-c1ccc(CNC(=O)[C@@H]2C[C@@H](O)CN2C(=O...,BRD4,O60885,VHL,726,78,Degradation of BRD4 in A375 cells after 4 h tr...,positive,1


shape: (2254, 11)

label counts:
Label
1    1436
0     818
Name: count, dtype: int64
missing uniprot id: 54


### 1. data splitting
create two sets of train/validation/test assignments:
- molecule-grouped reference
  - grouping: same canonical smiles stays in one paritition
  - purpose: prevent repeated observations of same molecule being in different partitions
- held-out-target
  - grouping: same uniprot id in same partition
  - purpose: test prediction on targets not used in training

In [4]:
# preserve missing uniprot id rows for later to review
missing_uniprot = protac_clean.loc[protac_clean["Uniprot"].isna()].copy()

# uniprot id is present, use for data split
split_data = protac_clean.loc[protac_clean["Uniprot"].notna()].copy()

# save both as csv in processed data folder

processed_dir = project_root / "data" / "processed"

missing_uniprot.to_csv(processed_dir / "missing_uniprot.csv", index = False)
split_data.to_csv(processed_dir / "split_data.csv", index = False)

print("set aside:", len(missing_uniprot))
print("available for splitting:", len(split_data))
print("\nlabel counts:")
print(split_data["Label"].value_counts())

set aside: 54
available for splitting: 2200

label counts:
Label
1    1399
0     801
Name: count, dtype: int64


In [5]:
# molecule-grouped reference split

molecule_split = split_data.copy()

# seven folds: 5 folds for trianing, 1 fold for validation, 1 fold for testing
sgkf = StratifiedGroupKFold(n_splits = 7, shuffle = True, random_state= 42)
molecule_split["split"] = "train"

# _ = ignoring first array, fold_indices = current held-out fold's row positions
#.split returns 2 arrays (train_indices, test_indices)
for fold, (_, test_indices) in enumerate(
    sgkf.split(
        X = molecule_split,
        y = molecule_split["Label"],
        groups =molecule_split["Canonicalised Smiles"]
    )
):
    if fold == 0: 
        molecule_split.iloc[
            test_indices, molecule_split.columns.get_loc("split")
        ] = "test"

    elif fold == 1:
        molecule_split.iloc[
            test_indices, molecule_split.columns.get_loc("split")
        ] = "validation"

# make sure each molecular structure belongs to one partition
assert(
    molecule_split.groupby("Canonicalised Smiles")["split"].nunique().max() == 1
), "a molecule appears in multiple partitions"

#inspect
print("rows per partition:")
print(molecule_split["split"].value_counts())

print("\nlabel counts per partition:")
print(pd.crosstab(molecule_split["split"], molecule_split["Label"]))

print("\nlabel proportions per partition:")
print(
    pd.crosstab(
        molecule_split["split"],
        molecule_split["Label"],
        normalize="index"
    ).round(3)
)

# save the dataframe with split assignments
processed_dir = project_root / "data" / "processed"
molecule_split.to_csv(processed_dir / "molecule_split.csv", index=False)

rows per partition:
split
train         1572
validation     314
test           314
Name: count, dtype: int64

label counts per partition:
Label         0    1
split               
test        114  200
train       573  999
validation  114  200

label proportions per partition:
Label           0      1
split                   
test        0.363  0.637
train       0.365  0.635
validation  0.363  0.637


In [10]:
# molecule-grouped split verification
assert len(molecule_split) == 2200, " molecule split dataframe does not have expected row count"

In [6]:
# held-out target split

target_split = split_data.copy()
target_split["split"] = "train"

# seven folds: five for training, one for validation, one for testing
target_splitter = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=42)

for fold, (_, test_indices) in enumerate(
    target_splitter.split(
        X = target_split,
        y = target_split["Label"],
        groups= target_split["Uniprot"]
    )
):
    if fold == 0:
        target_split.iloc[
            test_indices, target_split.columns.get_loc("split")
        ] = "test"

    elif fold == 1:
        target_split.iloc[
            test_indices, target_split.columns.get_loc("split")
        ] = "validation"

# verify that each target belongs to only one partition
assert (
    target_split.groupby("Uniprot")["split"].nunique().max() == 1
), "a target appears in multiple partitions"

# inspect partition sizes, target counts, and class balance
print("rows per partition:")
print(target_split["split"].value_counts())

print("\nunique targets per partition:")
print(target_split.groupby("split")["Uniprot"].nunique())

print("\nlabel counts per partition:")
print(pd.crosstab(target_split["split"], target_split["Label"]))

print("\nlabel proportions per partition:")
print(
    pd.crosstab(
        target_split["split"],
        target_split["Label"],
        normalize="index"
    ).round(3)
)

# report molecule overlap; target separation does not prevent this
train_molecules = set(target_split.loc[target_split["split"] == "train", "Canonicalised Smiles"])

test_molecules = set(target_split.loc[target_split["split"] == "test", "Canonicalised Smiles"])

print("\nshared molecules between train and test:", len(train_molecules & test_molecules))

# save checked target-held-out assignments
processed_dir = project_root / "data" / "processed"
target_split.to_csv(
    processed_dir / "target_split.csv",
    index=False
)

print("\ntarget separation verified; saved target_split.csv")

rows per partition:
split
train         1565
validation     325
test           310
Name: count, dtype: int64

unique targets per partition:
split
test           19
train         114
validation     10
Name: Uniprot, dtype: int64

label counts per partition:
Label         0    1
split               
test        112  198
train       568  997
validation  121  204

label proportions per partition:
Label           0      1
split                   
test        0.361  0.639
train       0.363  0.637
validation  0.372  0.628

shared molecules between train and test: 35

target separation verified; saved target_split.csv


In [9]:
# target split verification
assert len(target_split) == 2200, "target split dataframe does not have expected row count"

### checkpoint: evaluation splits complete

- started with 2,254 ribes-labeled rows.
- set aside 54 rows missing uniprot ids; used the same 2,200 rows for both splitting strategies.
- used `stratifiedgroupkfold` with 7 folds, shuffle enabled, and `random_state=42`.
- assigned fold 0 to test, fold 1 to validation, and folds 2–6 to training.

#### molecule-grouped reference split

- grouped by canonical smiles; verified no identical structures cross partitions.
- evaluates predictions on held-out molecules, not necessarily unseen targets.

| partition | rows | negative | positive |
|---|---:|---:|---:|
| train | 1,572 | 573 | 999 |
| validation | 314 | 114 | 200 |
| test | 314 | 114 | 200 |

#### held-out-target split

- grouped by uniprot; verified no target ids cross partitions.
- evaluates predictions on targets absent from training.

| partition | rows | targets | negative | positive |
|---|---:|---:|---:|---:|
| train | 1,565 | 114 | 568 | 997 |
| validation | 325 | 10 | 121 | 204 |
| test | 310 | 19 | 112 | 198 |

- 35 unique molecular structures overlap between training and test; retained because this experiment holds out targets, not molecules.
- target-id separation does not guarantee separation of related protein families.

#### saved outputs

- `data/processed/molecule_split.csv`
- `data/processed/target_split.csv`

use these fixed assignments for all model comparisons. validation guides model selection; test remains reserved for final evaluation.